# 📗 부록: 동기·비동기(`async`/`await`) 기초

> **이 노트북은 수업 시간에 다루지 않는 참고 자료입니다.** 완주 기준에 들어가지 않습니다. `async`·`await` 라는 낱말이 눈에 걸릴 때 열어 보세요.

파이썬 코드를 읽다 보면 이렇게 생긴 것을 만납니다.

```python
async def fetch(url):
    result = await get(url)
    return result
```

`def` 앞의 **`async`** 와 그 안의 **`await`** 가 무엇인지, 이 과정에서는 다룬 적이 없습니다. 이 부록은 **동기**와 **비동기**가 무엇이 다른지 처음부터 풀어 봅니다.

**이 부록에서 하는 것**

- [ ] **동기** 실행: 앞이 끝나야 뒤가 시작한다는 것을 시간으로 확인한다
- [ ] 기다리는 시간의 정체: **일하느라 느린 것**과 **기다리느라 느린 것**은 다르다
- [ ] **`async def`** 로 만든 함수는 **불러도 실행되지 않는다**(코루틴 객체)
- [ ] **`await`** 로 그 주문서를 실제로 실행해 값을 꺼낸다
- [ ] **`asyncio.gather`** 로 여러 개를 **동시에** 기다려 3초를 1초로 줄인다
- [ ] 주피터와 `.py` 에서 **시작하는 방법이 다르다**는 것을 안다
- [ ] 비동기가 **위로 전염된다**는 것. 부르는 쪽도 비동기여야 한다
- [ ] **`asyncio.Semaphore`** 로 동시에 나가는 호출 수를 제한한다
- [ ] LangChain 체인의 **`ainvoke`·`abatch`** 가 같은 원리로 돈다는 것을 확인한다

---
# 1. 동기: 앞이 끝나야 뒤가 시작한다

지금까지 우리가 쓴 코드는 전부 **동기(synchronous)** 였습니다. 위에서 아래로, **한 줄이 끝나야 다음 줄**이 시작합니다. 너무 당연해서 이름을 붙일 일도 없었습니다.

그 당연함이 손해가 되는 자리가 있습니다. **기다리는 일**이 여럿일 때입니다. `time.sleep(1)` 을 "1초 걸리는 일" 이라고 치고 세 개를 처리해 봅니다.

In [ ]:
import time


def slow_job(name):
    """1초 걸리는 일 하나를 흉내 낸다(실제로는 네트워크 응답을 기다리는 시간)."""
    time.sleep(1)                       # 1초 동안 아무것도 하지 않고 멈춰 있는다
    return f'{name} 완료'


# perf_counter 는 '지금 몇 초인가' 를 재는 시계다 - 시작과 끝을 각각 재서 뺀다
start = time.perf_counter()             # 시작 시각

for job in ('A', 'B', 'C'):             # 세 개를 차례로 처리한다
    print(slow_job(job))

end = time.perf_counter()               # 끝 시각
print(f'걸린 시간: {end - start:.1f}초')   # 끝 - 시작 = 실제로 걸린 시간

**3초**가 나옵니다. 1초짜리 일 세 개를 **차례로** 기다렸으니 당연합니다.

그런데 이 3초 동안 컴퓨터가 **바빴을까요?** 아닙니다. `time.sleep(1)` 은 계산을 하는 게 아니라 **아무것도 하지 않고 멈춰 있는** 명령입니다.

---
# 2. 느린 데는 두 종류가 있다

| | 무엇 | 예 | 기다리는 동안 CPU 는 |
|---|---|---|---|
| **일하느라 느리다** | 계산 자체가 오래 걸림 | 큰 표 정렬, 모델 학습 | **바쁘다** |
| **기다리느라 느리다** | 남의 답을 기다림 | 웹 요청, 파일 읽기, DB 응답 | **논다** |

비동기는 **아래쪽 한 줄**을 위한 장치입니다. 계산을 빠르게 해 주지 않습니다. 대신 **기다리는 동안 다른 일을 시작**하게 해 줍니다.

<div style="margin: 32px 0 36px;"><img src="images/sync_vs_async_timeline.png" width="820"></div>

웹에서 자료를 받아 오는 일이 정확히 아래쪽입니다. 요청을 보내 놓고 **답이 올 때까지 기다립니다.** 그 기다림이 열 번이면 열 번을 차례로 기다릴 이유가 없습니다.

## 여러 일을 동시에 '하는' 것은 아니다

여기서 오해하기 쉬운 지점이 있습니다. 비동기를 쓴다고 **두 가지 일이 함께 진행되지는 않습니다.** 일하는 사람은 여전히 **한 명(스레드 하나)** 입니다. 그 한 명이 **기다리는 자리에서만** 손을 놓고 다른 일로 옮겨 갈 뿐입니다.

> 비동기는 **`await` 를 만난 자리에서만** 다른 일로 옮겨 가므로, **기다림은 겹쳐도 계산은 겹치지 못합니다**(뒤에서 시간으로 확인합니다).

## 말 정리: 프로세서·프로세스·스레드

<div style="margin: 32px 0 36px;"><img src="images/process_thread.png" width="820"></div>

| 말 | 한 줄 |
|---|---|
| **프로세서(CPU)** | 계산을 실제로 하는 부품. **코어 개수만큼** 계산을 진짜로 한꺼번에 한다 |
| **프로세스** | 실행 중인 프로그램 한 벌. 메모리를 자기 것으로 따로 가진다(주피터 커널이 프로세스 하나) |
| **스레드** | 프로세스 안에서 코드를 한 줄씩 실행해 나가는 흐름. 우리 노트북은 **스레드 하나**로 돈다 |

## 여러 일을 동시에 처리하는 방법 세 가지

<div style="margin: 32px 0 36px;"><img src="images/concurrency_three_ways.png" width="820"></div>

| 방식 | 어떻게 동시에 처리하나 | 언제 쓰나 |
|---|---|---|
| **멀티프로세싱** | 프로세스를 여러 개로 나눠 **코어마다 하나씩** 맡긴다. 여러 계산이 **같은 시각에** 진행된다 | **계산**이 느릴 때 |
| **멀티스레딩** | 한 프로세스 안 스레드 여러 개가 **번갈아** 돈다. 갈아탈 시점은 운영체제가 정한다 | **기다림**이 많을 때 |
| **비동기** | 스레드는 하나 그대로, **`await` 를 만난 자리에서 번갈아** 돈다. 갈아탈 지점을 내가 정한다 | **기다림**이 많을 때 |

셋 중 **같은 시각에 계산이 둘 이상 도는 것은 멀티프로세싱뿐**입니다. 나머지 둘은 한 번에 하나씩, 빠르게 번갈아 할 뿐입니다.

> 고르는 기준 한 줄: **계산이 느리면 멀티프로세싱, 기다림이 느리면 나머지 둘.** 그 둘 중에서는 **쓰려는 라이브러리가 비동기 API(`async def` 로 정의되어 `await` 로 부르는 함수)를 제공하면 비동기, 부르면 끝날 때까지 멈추는 동기 API 뿐이면 멀티스레딩**입니다.

> 실무에서는 **비동기를 가장 많이 씁니다.** 하는 일이 대개 바깥 서비스를 여럿 부르는 것(웹 API·데이터베이스·LLM 호출)이고, 요즘 라이브러리는 대부분 비동기 API 를 함께 제공하기 때문입니다. 동기 API 만 있는 라이브러리를 쓸 때만 `asyncio.to_thread` 로 스레드에 넘기고, 멀티프로세싱은 무거운 계산에서 가끔 씁니다(계산이 느릴 때도 파이썬에서는 넘파이·판다스의 벡터 연산이나 라이브러리의 병렬 옵션을 먼저 봅니다).

---
# 3. 코루틴: 멈췄다가 그 자리에서 이어 가는 함수

> **코루틴(coroutine)** 은 **실행 도중에 스스로 멈췄다가, 나중에 멈춘 그 줄부터 다시 이어서 실행할 수 있는 함수**입니다.

우리가 지금까지 쓴 함수는 그럴 수 없었습니다. 한 번 부르면 `return` 을 만날 때까지 **중간에 멈추지 않고 끝까지 달립니다.**

| | 일반 함수 | 코루틴 |
|---|---|---|
| 정의 | `def` | **`async def`** |
| 부르면 | 곧바로 실행되어 **결과**가 나온다 | 실행되지 않고 **코루틴 객체**가 나온다 |
| 중간에 멈추기 | 못 한다 | **`await` 를 만나면 멈추고 자리를 비켜 준다** |
| 다시 이어 가기 | 처음부터 다시 부르는 수밖에 없다 | **멈춘 그 줄부터** 이어 간다 |

이 "멈췄다 이어 가기" 가 비동기의 전부입니다. 기다려야 하는 자리에서 멈추고 자리를 비켜 주니, 그동안 다른 코루틴이 자기 몫을 진행할 수 있습니다.

## 만들어 보기

`def` 앞에 **`async`** 를 붙이면 코루틴이 됩니다. 그리고 `time.sleep` 대신 **`asyncio.sleep`** 을 씁니다. 이름은 비슷하지만 하는 일이 다릅니다. `time.sleep` 은 **모두를 멈춰 세우고**, `asyncio.sleep` 은 **"나는 기다릴 테니 그동안 다른 일 하세요"** 라고 자리를 비켜 줍니다.

In [ ]:
import asyncio


async def slow_job_async(name):
    """1초 걸리는 일 하나. await 를 만나면 그 자리에서 멈추고 자리를 비켜 준다."""
    await asyncio.sleep(1)     # '1초 기다리는 동안 다른 일 하세요'
    return f'{name} 완료'      # 1초 뒤, 멈췄던 이 줄부터 이어 간다


# 괄호를 붙여 '불렀는데' 결과 문자열이 아니라 낯선 객체가 나온다
result = slow_job_async('A')
print(type(result).__name__, '->', result)

<div style="margin: 32px 0 36px;"><img src="images/coroutine_call.png" width="820"></div>

`slow_job_async('A')` 가 돌려준 것은 **코루틴 객체**입니다. **"할 일을 적어 둔 주문서"** 라고 생각하면 됩니다. 주문서를 써 두었을 뿐, 아직 아무도 그 일을 시작하지 않았습니다.

> **말 정리**: `async def` 로 정의한 것을 **코루틴 함수**, 그것을 불러 나온 주문서를 **코루틴 객체**라고 합니다. 그냥 "코루틴" 이라고 하면 보통 **객체** 쪽을 가리킵니다.

---
# 4. `await`: 주문서를 실행해 값을 꺼낸다

그 주문서를 실제로 실행하고 **결과가 나올 때까지 기다렸다가 값을 꺼내는** 것이 **`await`** 입니다.

> **주피터 노트북에서는 셀에 `await` 를 그냥 쓸 수 있습니다.** 주피터가 셀을 비동기로 감싸 돌려 주기 때문입니다. `.py` 파일에서는 이렇게 쓸 수 없습니다. 뒤에서 다룹니다.

In [ ]:
start = time.perf_counter()             # 시작 시각

# 앞 절에서 만들어 두기만 했던 그 주문서(result)를 이제 실행한다
print(await result)

end = time.perf_counter()               # 끝 시각 - 주문서 처리가 끝난 뒤의 시각이다
print(f'걸린 시간: {end - start:.1f}초')

1초가 걸리고 문자열이 나왔습니다. **여기까지는 동기와 똑같습니다**. 하나만 기다렸으니까요. 비동기의 이득은 **여러 개를 기다릴 때** 나옵니다.

---
# 5. `asyncio.gather`: 동시에 기다리기

주문서를 **세 장 만들어 한꺼번에 넣으면**, 셋이 함께 처리됩니다. 그 문법이 **`asyncio.gather`** 입니다.

먼저 **잘못 쓰는 법**부터 봅니다. `await` 를 세 번 따로 쓰면 비동기로 적어 놓고도 3초가 걸립니다.

In [ ]:
# 한 번에 하나씩 await - 비동기로 적었지만 결국 차례로 기다린다
start = time.perf_counter()             # 시작 시각

a = await slow_job_async('A')           # 여기서 1초를 다 기다리고 나서야
b = await slow_job_async('B')           # 다음 줄이 시작한다
c = await slow_job_async('C')

print(a, b, c)

end = time.perf_counter()               # 끝 시각
print(f'하나씩 await: {end - start:.1f}초')

In [ ]:
# gather 로 한꺼번에 - 주문서 세 장을 함께 넣는다
start = time.perf_counter()             # 시작 시각

# 괄호를 붙여 만들기만 한 코루틴 세 개를 gather 에 넘긴다(여기서는 await 를 붙이지 않는다)
results = await asyncio.gather(
    slow_job_async('A'),
    slow_job_async('B'),
    slow_job_async('C'),
)

print(results)                          # 넘긴 순서 그대로 결과가 담긴다

end = time.perf_counter()               # 끝 시각 - 셋이 모두 끝난 뒤의 시각이다
print(f'gather: {end - start:.1f}초')

**3초가 1초가 됐습니다.** 일을 빨리 한 게 아니라, **기다리는 시간을 겹쳐 놓은** 것입니다.

> `gather` 가 돌려주는 목록의 순서는 **넘긴 코루틴의 순서**와 같습니다. 먼저 끝난 것이 앞으로 오지 않습니다. 위에서 `results[0]` 은 0.3초 만에 끝났든 아니든 언제나 첫 번째로 넘긴 `slow_job_async('A')` 의 결과입니다. 그래서 결과를 입력과 짝지을 때 따로 이름표를 붙일 필요가 없습니다.

## gather 로도 줄지 않는 것: 계산

앞에서 말한 것을 이제 시간으로 확인합니다. **기다림이 없는 순수 계산**을 `gather` 로 묶으면 어떻게 될까요? 위의 3초가 1초가 된 것처럼 줄어들까요?

In [ ]:
def heavy(n):
    """0부터 n-1 까지 제곱을 더한다 - 기다리는 시간 없이 CPU 가 계속 일한다."""
    return sum(i * i for i in range(n))


async def heavy_async(n):
    """계산만 하는 코루틴 - 안에 await 가 없어 도중에 자리를 비켜 줄 지점이 없다."""
    return heavy(n)


N = 10_000_000                          # 몇 백 밀리초쯤 걸리는 계산량

start = time.perf_counter()             # 시작 시각
await heavy_async(N)                    # 계산 하나
end = time.perf_counter()               # 끝 시각
one = end - start
print(f'계산 하나: {one:.2f}초')

start = time.perf_counter()             # 시작 시각
await asyncio.gather(                   # 같은 계산 셋을 gather 로 묶는다
    heavy_async(N),
    heavy_async(N),
    heavy_async(N),
)
end = time.perf_counter()               # 끝 시각
print(f'계산 셋 gather: {end - start:.2f}초 (하나의 약 3배)')

**전혀 줄지 않습니다.** 셋을 묶었더니 하나의 약 3배가 걸립니다. `asyncio.sleep` 을 묶었을 때 3초가 1초가 된 것과 정반대입니다.

이유는 하나입니다. `heavy_async` 안에는 **`await` 가 없습니다.** 자리를 비켜 줄 지점이 없으니 첫 번째 계산이 끝까지 CPU 를 붙들고, 그다음에야 두 번째가 시작합니다. **겹칠 기다림이 없으면 겹칠 것도 없습니다.**

---
# 6. 비동기는 위로 전염된다

`await` 에는 규칙이 하나 있습니다. **코루틴 안에서만 쓸 수 있습니다.** 평범한 `def` 안에서 쓰면 실행 이전에 **문법 오류**로 걸립니다.

그래서 코루틴을 하나 쓰기 시작하면 **그것을 부르는 함수도 코루틴이어야 하고**, 또 그것을 부르는 함수도 코루틴이어야 합니다. 비동기는 이렇게 **부르는 쪽으로 번져 올라갑니다.**

먼저 **안 되는 코드**입니다. 평범한 `def` 안에서 `await` 를 썼습니다.

```python
def normal():
    return await slow_job_async('A')
```

이 두 줄을 셀에 넣고 실행하면 함수를 부르기도 전에 이렇게 걸립니다.

```
SyntaxError: 'await' outside async function
```

**실행 중에 난 오류가 아니라 문법 오류**입니다. 파이썬이 셀을 읽는 단계에서 걸러 냅니다. 직접 보고 싶으면 새 셀에 위 두 줄만 붙여 실행해 보세요. 그 셀은 통째로 오류가 나므로 다른 코드와 같은 셀에 두면 안 됩니다.

고치는 방법은 하나입니다. **`def` 를 `async def` 로 바꾸는 것**입니다. 그러면 이 함수도 코루틴이 되고, 이 함수를 부르는 쪽도 `await` 를 써야 합니다.

In [ ]:
# async def 안에서는 await 를 쓸 수 있다 - 그래서 부르는 쪽도 async 가 된다
async def report():
    """slow_job_async 를 await 하려니 이 함수도 코루틴이어야 한다."""
    done = await slow_job_async('A')      # 여기서 1초 기다렸다가 결과를 받는다
    return f'보고: {done}'

In [ ]:
# 정의만 해 두면 아무 일도 일어나지 않는다 - await 로 실행해야 값이 나온다
print(await report())

함수가 세 겹이면 세 겹 모두 `async def` 가 됩니다. 그러면 **맨 바깥은 누가 부르나요?** 거기서 한 번, 이벤트 루프를 켜 줘야 합니다. 그게 다음 절입니다.

---
# 7. 주피터와 `.py` 는 시작하는 방법이 다르다

비동기 코드는 **이벤트 루프**라는 진행자 위에서 돕니다. 주문서들을 받아 두고 "이건 기다리는 중이니 저것부터" 하고 번갈아 처리해 주는 역할입니다.

차이는 **그 진행자를 누가 켜는가** 하나입니다.

| | 이벤트 루프 | 최상위 `await` | `asyncio.run(...)` |
|---|---|---|---|
| **주피터 노트북** | 커널이 **이미 켜 둠** | ✅ 그냥 쓴다(위 셀들) | ❌ `RuntimeError`: 이미 도는 루프 위에서는 못 켠다 |
| **`.py` 파일** | 없음. 내가 켠다 | ❌ `SyntaxError` | ✅ 이렇게 시작한다 |

그래서 같은 코드를 `.py` 로 옮기면 **`async def main()` 으로 감싸고 `asyncio.run(main())` 으로 시작**해야 합니다.

```python
# hello_async.py 로 저장하고 터미널에서 실행할 때의 모양
import asyncio

async def main():
    results = await asyncio.gather(slow_job_async('A'), slow_job_async('B'))
    print(results)

asyncio.run(main())     # 여기서 이벤트 루프를 켜고, 끝나면 끈다
```

> **`asyncio.run` 은 프로그램에서 딱 한 번, 가장 바깥에서 부릅니다.** 안쪽 함수들끼리는 `await` 로 서로를 부릅니다.

### 🖐️ 함께 따라하기: 걸린 시간을 직접 재 보기

기다리는 시간이 **제각각일 때** gather 가 얼마나 걸리는지 확인합니다.

1. `async def wait_for(name, seconds)` 를 만드세요. `await asyncio.sleep(seconds)` 로 기다린 뒤 `f'{name}({seconds}초)'` 를 돌려줍니다.
2. `start = time.perf_counter()` 로 시작 시각을 재고, `asyncio.gather` 로 **`('짧게', 0.3)`·`('보통', 0.6)`·`('길게', 1.0)`** 세 개를 한꺼번에 기다리세요.
3. 끝난 뒤 `end = time.perf_counter()` 를 재서, 결과 목록과 **`end - start`** 를 소수 첫째 자리까지 출력하세요.

**확인 기준**: 결과가 `['짧게(0.3초)', '보통(0.6초)', '길게(1.0초)']` 처럼 **넘긴 순서대로** 나오고, 걸린 시간이 **세 시간의 합(1.9초)이 아니라 가장 긴 하나(약 1.0초)** 입니다. 여럿을 함께 기다리면 **가장 오래 걸리는 하나만큼**만 걸린다는 뜻입니다.

In [ ]:
# 여기에 코드를 작성하세요
# 1) async def wait_for(name, seconds) 를 만든다 (await asyncio.sleep(seconds) 후 문자열 반환)
# 2) start = time.perf_counter() 로 시작 시각을 재고, asyncio.gather 로
#    ('짧게', 0.3)·('보통', 0.6)·('길게', 1.0) 을 함께 기다린다
# 3) end = time.perf_counter() 로 끝 시각을 재고, 결과 목록과 end - start 를
#    소수 첫째 자리까지 출력한다

---
# 8. 어디에 쓰고, 어디에 쓰지 않나

라이브러리에서 **같은 기능이 두 벌**로 제공되는 경우를 자주 만납니다. 이름 앞이나 뒤에 **`a`**(async) 나 `_async` 가 붙은 쪽이 **비동기 API**, 곧 코루틴 함수입니다.

| 동기 | 비동기 | 무엇을 기다리나 |
|---|---|---|
| `requests.get(url)` | `session.get(url)`(`await`) | 웹 서버의 응답 |
| `conn.execute(sql)` | `await conn.execute(sql)` | 데이터베이스의 응답 |
| `obj.invoke(...)` | `await obj.ainvoke(...)` | 바깥 서비스의 응답 |

고르는 기준은 하나입니다. **여러 개를 동시에 기다릴 일이 있는가.**

- 요청이 **하나뿐**이면 비동기로 얻는 것이 없습니다. 1초는 그대로 1초입니다(앞에서 본 그대로).
- 요청이 **여럿**이고 서로 **순서에 상관없으면** 비동기가 크게 이깁니다(앞의 3초 → 1초).
- **계산**이 느린 것이라면 비동기는 답이 아닙니다(2절 표의 위쪽).

> 그리고 비동기는 공짜가 아닙니다. 앞에서 봤듯 **부르는 쪽 전부가 코루틴이 되어야** 합니다. 그래서 "기다림이 여럿" 이 아닌 곳에서는 굳이 쓰지 않습니다.

---
# 9. 한꺼번에 맡기되, 동시에 나가는 수는 제한한다

실무에서 `gather` 를 그대로 쓰다 걸리는 자리가 있습니다. 요청 100개를 한 번에 넘기면 **100개가 동시에 나갑니다.** 받는 쪽은 대개 그것을 반기지 않습니다. 상용 API 는 **분당 호출 한도**를 두고 넘치면 `429 Too Many Requests` 로 끊고, 우리 쪽도 연결이 한꺼번에 열리며 불안정해집니다.

그래서 **일은 전부 맡기되 동시에 진행하는 수만 묶어 두는** 장치를 씁니다. `asyncio.Semaphore(n)` 입니다. **동시에 n 개만 통과시키는 문지기**라고 보면 됩니다. 자리가 차 있으면 다음 코루틴은 문 앞에서 기다리다가, 하나가 끝나 자리가 나면 들어갑니다.

In [ ]:
# 동시에 3개까지만 통과시키는 문지기 - 세 함수가 이 하나를 함께 본다
sem = asyncio.Semaphore(3)


async def fetch_news(topic):
    """뉴스 API 를 부르는 흉내. 응답까지 0.5초."""
    async with sem:                     # 자리가 없으면 이 줄에서 기다린다
        await asyncio.sleep(0.5)
    return f'뉴스({topic})'              # 블록을 벗어나면 자리를 내놓는다


async def fetch_weather(city):
    """날씨 API 를 부르는 흉내. 응답까지 0.4초."""
    async with sem:
        await asyncio.sleep(0.4)
    return f'날씨({city})'


async def fetch_rate(pair):
    """환율 API 를 부르는 흉내. 응답까지 0.6초."""
    async with sem:
        await asyncio.sleep(0.6)
    return f'환율({pair})'


# 서로 다른 호출 여덟 개를 한꺼번에 맡긴다
jobs = [
    fetch_news('AI'), fetch_news('경제'), fetch_news('스포츠'),
    fetch_weather('서울'), fetch_weather('부산'),
    fetch_rate('USD/KRW'), fetch_rate('EUR/KRW'), fetch_rate('JPY/KRW'),
]

start = time.perf_counter()             # 시작 시각
results = await asyncio.gather(*jobs)   # 별표는 목록을 인자 8개로 펼쳐 넘긴다
end = time.perf_counter()               # 끝 시각

print(results)
print(f'8개 호출, 동시에 3개까지: {end - start:.1f}초')

**약 1.5초**가 나옵니다. 제한이 없었다면 여덟 개가 함께 나가 가장 긴 0.6초에 끝났을 것입니다. **일부러 느리게 만든 것**입니다. 그 대신 서버가 한 번에 받는 요청이 항상 3개 이하로 유지됩니다.

함수가 셋이지만 **문지기는 하나**라는 점을 보세요. 세마포어는 함수별이 아니라 **같은 곳으로 나가는 호출 전체**를 묶습니다. 뉴스든 날씨든 환율이든 합쳐서 3개까지입니다.

| | 동시에 나가는 요청 | 걸린 시간 | 결과 |
|---|---|---|---|
| `gather` 만 | 8개 전부 | 0.6초 | 개수가 커지면 `429`·연결 실패 |
| `gather` + `Semaphore(3)` | 항상 3개 이하 | 1.5초 | 끝까지 안정적으로 마친다 |

> 요청 수가 많아질수록 이 차이가 커집니다. **`gather` 는 일을 맡기는 도구, 세마포어는 동시에 몇 개까지 내보낼지 정하는 도구**입니다. 둘은 함께 씁니다.

## `async with sem:` 을 어디에 두는가

1. **세마포어는 함수 바깥에 하나만.** 모두가 같은 문지기를 봐야 수가 세어집니다.
2. **`async with sem:` 은 코루틴 안에.** 함수 하나가 곧 요청 하나입니다.
3. **블록 안에는 기다리는 `await` 만.** 자리를 잡고 있는 동안 다른 요청은 못 들어옵니다.

```python
sem = asyncio.Semaphore(3)           # 문지기는 함수 바깥에 하나

async def job(name):
    # ❌ 뒷정리까지 블록 안에 두면 그동안 자리를 붙들고 있는다
    async with sem:
        data = await call_api(name)
        save_to_file(parse(data))    # 기다림이 아닌 일

    # ✅ 기다리는 호출만 감싸고, 나머지는 자리를 내놓고 한다
    async with sem:
        data = await call_api(name)
    return parse(data)
```

> 실무에서는 **상대가 정한 한도**를 그대로 n 으로 씁니다(예: 분당 60회 제한이면 여유를 두고 5~10). 라이브러리가 자체 한도 설정을 제공하면 그것을 먼저 보고, 없을 때 세마포어로 감쌉니다.

---
# 10. 실무에서 만나는 자리: LangChain 체인

여기까지가 파이썬 문법입니다. 마지막으로 **이 문법이 실제 라이브러리에서 어떤 이름으로 보이는지** 봅니다. LangChain 체인은 앞의 규칙 그대로, 동기 메서드 옆에 **`a` 가 붙은 비동기 메서드**를 함께 가집니다.

| 하나 처리 | 여러 개 처리 |
|---|---|
| `chain.invoke(입력)` | `chain.batch([입력, 입력, ...])` |
| `await chain.ainvoke(입력)` | `await chain.abatch([입력, 입력, ...])` |

> **이 절만 실제 모델을 부릅니다.** 다른 노트북과 같은 `OPENAI_API_KEY` 가 필요합니다(요청 아홉 번, 문장 한 줄짜리라 비용은 아주 적습니다). 앞의 1~9절은 키 없이 그대로 돌아갑니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 절은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료")

체인은 지금까지 쓰던 것과 똑같이 만듭니다. **부르는 메서드만 `ainvoke`** 입니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
prompt = ChatPromptTemplate.from_template('다음 문장을 15자 이내 한 줄로 요약하세요.\n\n문장: {text}')
chain = prompt | model | StrOutputParser()   # 프롬프트 -> 모델 -> 문자열

start = time.perf_counter()             # 시작 시각
answer = await chain.ainvoke({'text': '오늘 서울 지점 매출이 지난주보다 12퍼센트 늘었다.'})
end = time.perf_counter()               # 끝 시각

print(answer)
print(f'ainvoke 하나: {end - start:.1f}초')

이제 **네 문장을 요약**시킵니다. 하나씩 `await` 한 경우와 `abatch` 로 한꺼번에 맡긴 경우를 같은 셀에서 재 봅니다. 앞에서 본 그 차이가 실제 모델 호출에서도 그대로 나타납니다.

In [ ]:
inputs = [
    {'text': '서울 지점 매출이 지난주보다 12퍼센트 늘었다.'},
    {'text': '부산 지점은 주말 방문객이 크게 줄었다.'},
    {'text': '대구 지점은 신메뉴 판매가 전체의 30퍼센트를 넘었다.'},
    {'text': '광주 지점은 배달 주문이 매장 주문을 앞질렀다.'},
]

# (1) 하나씩 await - 앞 요청이 끝나야 다음 요청을 보낸다
start = time.perf_counter()
one_by_one = [await chain.ainvoke(item) for item in inputs]
end = time.perf_counter()
print(f'ainvoke 네 번: {end - start:.1f}초')

# (2) gather - 코루틴 네 개를 만들어 한꺼번에 굴린다(앞에서 한 그대로)
start = time.perf_counter()
gathered = await asyncio.gather(*[chain.ainvoke(item) for item in inputs])
end = time.perf_counter()
print(f'gather 로 네 개: {end - start:.1f}초')

# (3) abatch - 같은 일을 체인이 대신해 준다(안에서 gather 와 같은 일을 한다)
start = time.perf_counter()
batched = await chain.abatch(inputs)
end = time.perf_counter()
print(f'abatch 한 번: {end - start:.1f}초')

for text, answer in zip(inputs, batched):
    print(f"{text['text'][:12]}... -> {answer}")

**하나씩만 유독 오래 걸립니다.** `gather` 와 `abatch` 는 둘 다 기다리는 시간을 겹쳐 놓아 그보다 훨씬 짧습니다. `abatch` 는 우리가 앞에서 손으로 하던 `gather` 를 체인이 대신해 주는 것이라 결과도 같습니다(실제 호출이라 초는 실행마다 오르내립니다).

앞의 **동시 호출 제한**도 라이브러리가 옵션으로 제공합니다. 세마포어를 직접 만들 필요 없이 `max_concurrency` 를 넘기면 됩니다. 상용 API 의 분당 한도에 걸리지 않게 할 때 씁니다.

In [ ]:
# 한 번에 2개까지만 나가게 제한한다(앞의 Semaphore(2) 와 같은 효과)
start = time.perf_counter()
limited = await chain.abatch(inputs, config={'max_concurrency': 2})
end = time.perf_counter()

print(limited)
print(f'abatch + max_concurrency=2: {end - start:.1f}초')   # 2개씩 두 묶음

제한 없는 `abatch` 보다 조금 더 걸립니다. 2개씩 두 묶음으로 나눠 보냈기 때문입니다. 앞에서 세마포어로 한 일을 **설정값 하나로** 한 셈입니다.

> 정리하면, 실무에서 쓰는 라이브러리는 대개 **`a` 가 붙은 비동기 메서드**와 **동시 실행 수 옵션**을 함께 제공합니다. 이 부록에서 본 `await`·`gather`·세마포어가 그 이름 뒤에서 하는 일입니다.

## 주의: 비동기로 부를 때 도구(tool)는

1. **동기 도구는 그대로 둬도 됩니다.** `@tool` 을 붙인 `def` 함수는 `ainvoke` 로 불러도 LangChain 이 스레드에서 돌려 줍니다.
2. **비동기 도구는 비동기로만 부릅니다.** `async def` 도구를 `invoke` 로 부르면 `NotImplementedError` 입니다. 도구 하나라도 비동기면 에이전트도 `ainvoke` 로 부릅니다.
3. **`async def` 안에서는 `await` 로 기다립니다.** `requests.get`·`time.sleep` 을 그대로 쓰면 그동안 다른 요청까지 멈춥니다.

```python
# ❌ 코루틴 안에서 동기 함수로 기다린다
async def fetch(url):
    return requests.get(url).text                            # 응답이 올 때까지 다른 요청도 함께 멈춘다

# ✅ 비동기 클라이언트로 await 하거나, 동기 함수뿐이면 스레드로 넘긴다
async def fetch(url):
    return (await asyncio.to_thread(requests.get, url)).text  # 기다리는 동안 다른 요청이 진행된다
```

> **2번의 대표가 MCP 도구입니다.** `get_tools()` 가 주는 도구는 코루틴만 있어 `await tool.ainvoke(...)` 로만 부릅니다.

---
## 정리

| 개념 | 핵심 |
|---|---|
| 동기 | 앞이 끝나야 뒤가 시작: 기다림이 여럿이면 그만큼 쌓인다 |
| 비동기가 돕는 곳 | **기다리느라** 느린 일(네트워크·파일·서버 응답). 계산은 아니다 |
| `async def` | 비동기 함수. **불러도 실행되지 않고** 코루틴 객체가 나온다 |
| `await` | 코루틴을 실제로 실행해 **값이 나올 때까지 기다린다**. 하나씩 쓰면 겹치지 않는다 |
| `asyncio.gather` | 여러 코루틴을 **함께** 기다린다. 가장 긴 하나만큼 걸린다 |
| `asyncio.run` | `.py` 에서 **가장 바깥에 한 번**. 주피터에서는 쓰지 않는다 |
| `asyncio.Semaphore(n)` | 맡기는 일은 그대로 두고 **동시에 나가는 수만 n 개로** 제한한다 |
| `ainvoke`·`abatch` | 라이브러리가 내놓는 비동기 API. `abatch` 는 `gather` 를 대신해 준다 |
| `a`/`_async` 이름 | 같은 기능의 비동기 API. 기다림이 여럿일 때 고른다 |

> 더 깊은 것들(`Task` 취소, `async for`, 스레드와의 비교, 이벤트 루프 직접 만들기)은 이 부록에서 일부러 다루지 않았습니다. **`async`·`await` 가 적힌 코드를 읽는 데는 여기까지면 충분합니다.**